# Signal Clustering dengan K-Means

Notebook ini melakukan clustering signal berdasarkan fitur statistik
dan fitur domain frekuensi.

Fitur yang digunakan:
- Mean
- Standard deviation
- Minimum dan maksimum
- Peak-to-peak amplitude
- RMS
- Envelope
- Zero crossing rate
- Dominant frequency
- Spectral centroid
- Spectral bandwidth
- Spectral energy
- SNR

Tahapan berikutnya adalah Elbow, Silhouette, K-Means, PCA,
Confusion Matrix, dan penyimpanan hasil.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import hilbert
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment

## 2. Konfigurasi

In [ ]:
SIGNAL_FILE = "data/signals.csv"
LABEL_COLUMN = "label"

SAMPLING_RATE = 1000

# None = semua kolom numerik selain label
SIGNAL_COLUMNS = None

K_MIN = 2
K_MAX = 8
SELECTED_K = None

RANDOM_STATE = 42

## 3. Ekstraksi Fitur Signal

Signal diubah menjadi beberapa fitur statistik, envelope,
dan fitur spektrum menggunakan FFT.

In [ ]:
def zero_crossing_rate(signal):
    signal = signal - np.mean(signal)
    crossings = np.where(np.diff(np.signbit(signal)))[0]
    return len(crossings) / max(len(signal) - 1, 1)


def extract_signal_features(signal, sampling_rate):
    signal = np.asarray(signal, dtype=float)
    signal = np.nan_to_num(signal, nan=np.nanmedian(signal))

    n = len(signal)

    mean_value = np.mean(signal)
    std_value = np.std(signal)
    min_value = np.min(signal)
    max_value = np.max(signal)
    peak_to_peak = max_value - min_value
    rms = np.sqrt(np.mean(signal ** 2))

    analytic_signal = hilbert(signal)
    envelope = np.abs(analytic_signal)

    envelope_mean = np.mean(envelope)
    envelope_std = np.std(envelope)
    envelope_max = np.max(envelope)

    zcr = zero_crossing_rate(signal)

    fft_values = np.fft.rfft(signal - mean_value)
    frequencies = np.fft.rfftfreq(n, d=1 / sampling_rate)
    magnitude = np.abs(fft_values)

    if len(magnitude) > 1:
        magnitude_no_dc = magnitude.copy()
        magnitude_no_dc[0] = 0

        dominant_index = np.argmax(magnitude_no_dc)
        dominant_frequency = frequencies[dominant_index]

        spectral_sum = np.sum(magnitude_no_dc) + 1e-12

        spectral_centroid = (
            np.sum(frequencies * magnitude_no_dc)
            / spectral_sum
        )

        spectral_bandwidth = np.sqrt(
            np.sum(
                ((frequencies - spectral_centroid) ** 2)
                * magnitude_no_dc
            ) / spectral_sum
        )

        spectral_energy = np.sum(magnitude_no_dc ** 2)
    else:
        dominant_frequency = 0
        spectral_centroid = 0
        spectral_bandwidth = 0
        spectral_energy = 0

    window = max(3, int(sampling_rate * 0.01))

    if window % 2 == 0:
        window += 1

    if window >= n:
        window = n if n % 2 == 1 else max(3, n - 1)

    if window >= 3:
        kernel = np.ones(window) / window
        smooth = np.convolve(signal, kernel, mode="same")
        noise = signal - smooth

        signal_power = np.mean(smooth ** 2) + 1e-12
        noise_power = np.mean(noise ** 2) + 1e-12

        snr_db = 10 * np.log10(signal_power / noise_power)
    else:
        snr_db = 0

    return [
        mean_value,
        std_value,
        min_value,
        max_value,
        peak_to_peak,
        rms,
        envelope_mean,
        envelope_std,
        envelope_max,
        zcr,
        dominant_frequency,
        spectral_centroid,
        spectral_bandwidth,
        spectral_energy,
        snr_db
    ]


FEATURE_NAMES = [
    "mean",
    "std",
    "min",
    "max",
    "peak_to_peak",
    "rms",
    "envelope_mean",
    "envelope_std",
    "envelope_max",
    "zero_crossing_rate",
    "dominant_frequency",
    "spectral_centroid",
    "spectral_bandwidth",
    "spectral_energy",
    "snr_db"
]

## 4. Load Dataset

Setiap baris dianggap sebagai satu signal.
Kolom numerik digunakan sebagai titik-titik signal.

In [ ]:
df = pd.read_csv(SIGNAL_FILE)

labels = df[LABEL_COLUMN] if LABEL_COLUMN in df.columns else None

if SIGNAL_COLUMNS is None:
    signal_columns = [
        col for col in df.columns
        if col != LABEL_COLUMN
        and pd.api.types.is_numeric_dtype(df[col])
    ]
else:
    signal_columns = SIGNAL_COLUMNS

if len(signal_columns) == 0:
    raise ValueError("Tidak ada kolom numerik untuk signal.")

print("Kolom signal:")
print(signal_columns)

## 5. Extract Signal Features

In [ ]:
feature_rows = []
valid_indices = []

for idx, row in df.iterrows():
    signal = row[signal_columns].values.astype(float)

    try:
        feature_rows.append(
            extract_signal_features(
                signal,
                SAMPLING_RATE
            )
        )
        valid_indices.append(idx)
    except Exception as e:
        print(f"Gagal memproses baris {idx}: {e}")

X_features = pd.DataFrame(
    feature_rows,
    columns=FEATURE_NAMES
)

df_valid = df.loc[valid_indices].reset_index(drop=True)

if labels is not None:
    labels = df_valid[LABEL_COLUMN]

print("Jumlah signal:", len(X_features))
print("Jumlah fitur:", X_features.shape[1])
display(X_features.head())

## 6. Imputation dan Standardisasi

Nilai tidak valid diganti menggunakan median, kemudian fitur
distandardisasi agar memiliki skala yang sebanding.

In [ ]:
X_features = X_features.replace(
    [np.inf, -np.inf],
    np.nan
)

X_features = X_features.fillna(
    X_features.median()
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

## 7. Elbow Method

In [ ]:
max_k = min(K_MAX, len(X_scaled) - 1)

if max_k < K_MIN:
    raise ValueError("Jumlah signal terlalu sedikit.")

k_values = range(K_MIN, max_k + 1)
inertias = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method - Signal Clustering")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Silhouette Score

In [ ]:
silhouette_results = {}

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    cluster_labels = model.fit_predict(X_scaled)

    silhouette_results[k] = silhouette_score(
        X_scaled,
        cluster_labels
    )

for k, score in silhouette_results.items():
    print(f"K={k}: {score:.4f}")

best_k = max(
    silhouette_results,
    key=silhouette_results.get
)

if SELECTED_K is None:
    SELECTED_K = best_k

print(f"\nK yang digunakan: {SELECTED_K}")

## 9. Final K-Means

In [ ]:
kmeans = KMeans(
    n_clusters=SELECTED_K,
    random_state=RANDOM_STATE,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

final_silhouette = silhouette_score(
    X_scaled,
    clusters
)

print(f"Silhouette Score final: {final_silhouette:.4f}")

## 10. PCA Visualization

In [ ]:
pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters,
    cmap="tab10",
    s=50
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA - Signal Clusters")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 11. Profil Fitur Setiap Cluster

Rata-rata fitur setiap cluster digunakan untuk melihat karakteristik
signal pada masing-masing kelompok.

In [ ]:
cluster_profile = X_features.copy()
cluster_profile["cluster"] = clusters

cluster_means = cluster_profile.groupby("cluster").mean()

display(cluster_means.round(4))

plt.figure(figsize=(12, 7))

sns.heatmap(
    cluster_means.T,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)

plt.xlabel("Cluster")
plt.ylabel("Signal Feature")
plt.title("Cluster Feature Profile")
plt.tight_layout()
plt.show()

## 12. Jumlah Signal Setiap Cluster

In [ ]:
unique, counts = np.unique(
    clusters,
    return_counts=True
)

for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} signal")

## 13. Confusion Matrix

Jika dataset memiliki label asli, cluster dipetakan ke label
menggunakan Hungarian Algorithm sebelum membuat confusion matrix.

In [ ]:
def best_cluster_label_mapping(y_true, y_cluster):
    true_labels = np.unique(y_true)
    cluster_labels = np.unique(y_cluster)

    true_to_int = {
        label: i for i, label in enumerate(true_labels)
    }

    y_true_int = np.array([
        true_to_int[x] for x in y_true
    ])

    n = max(
        len(true_labels),
        len(cluster_labels)
    )

    contingency = np.zeros(
        (n, n),
        dtype=int
    )

    for t, c in zip(y_true_int, y_cluster):
        contingency[t, c] += 1

    row_ind, col_ind = linear_sum_assignment(
        -contingency
    )

    mapping = {
        c: r for r, c in zip(row_ind, col_ind)
    }

    for c in cluster_labels:
        if c not in mapping:
            mapping[c] = 0

    y_pred_mapped = np.array([
        mapping[c] for c in y_cluster
    ])

    return (
        true_labels,
        y_true_int,
        y_pred_mapped
    )


if labels is not None and not labels.isna().all():

    valid = ~labels.isna()

    y_true = labels[valid].astype(str).values
    y_cluster = np.array(clusters)[valid]

    true_labels, y_true_int, y_pred_mapped = (
        best_cluster_label_mapping(
            y_true,
            y_cluster
        )
    )

    cm = confusion_matrix(
        y_true_int,
        y_pred_mapped,
        labels=np.arange(len(true_labels))
    )

    accuracy = accuracy_score(
        y_true_int,
        y_pred_mapped
    )

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=true_labels,
        yticklabels=true_labels
    )

    plt.xlabel("Predicted Cluster")
    plt.ylabel("Actual Label")

    plt.title(
        f"Confusion Matrix - Signal Clustering\n"
        f"Accuracy = {accuracy:.4f}"
    )

    plt.tight_layout()
    plt.show()

    print(
        pd.DataFrame(
            cm,
            index=true_labels,
            columns=true_labels
        )
    )

    print(f"\nAccuracy: {accuracy:.4f}")

else:
    print(
        "Label asli tidak tersedia. "
        "Confusion Matrix tidak dapat dibuat."
    )

## Helper Interpretasi Otomatis

Cell berikut membaca hasil clustering, silhouette score, PCA, dan confusion matrix lalu memberikan interpretasi otomatis.

In [ ]:
# Helper Interpretasi Hasil Signal Clustering

print("=" * 70)
print("HELPER INTERPRETASI SIGNAL CLUSTERING")
print("=" * 70)

# 1. Silhouette
print("\n1. INTERPRETASI SILHOUETTE SCORE")
print(f"Silhouette Score = {final_silhouette:.4f}")

if final_silhouette >= 0.70:
    print("Hasil: sangat baik.")
    print("Karakteristik signal antar cluster sangat berbeda.")
elif final_silhouette >= 0.50:
    print("Hasil: cukup baik.")
    print("Signal memiliki pemisahan cluster yang cukup jelas.")
elif final_silhouette >= 0.25:
    print("Hasil: sedang/lemah.")
    print("Sebagian signal masih memiliki karakteristik yang mirip.")
else:
    print("Hasil: kurang baik.")
    print("Pemisahan karakteristik signal antar cluster masih lemah.")

# 2. Cluster size
print("\n2. DISTRIBUSI SIGNAL")

cluster_counts = pd.Series(clusters).value_counts().sort_index()

for cluster_id, count in cluster_counts.items():
    percentage = count / len(clusters) * 100
    print(f"Cluster {cluster_id}: {count} signal ({percentage:.2f}%)")

# 3. Feature profile
print("\n3. FITUR DOMINAN SETIAP CLUSTER")

# Gunakan nilai z-score rata-rata agar perbandingan antar fitur lebih mudah
feature_z = pd.DataFrame(
    StandardScaler().fit_transform(cluster_means),
    index=cluster_means.index,
    columns=cluster_means.columns
)

for cluster_id in feature_z.index:
    row = feature_z.loc[cluster_id]
    highest = row.abs().sort_values(ascending=False).head(3).index

    print(f"Cluster {cluster_id}:")
    for feature in highest:
        direction = (
            "lebih tinggi"
            if row[feature] > 0
            else "lebih rendah"
        )
        print(
            f"  - {feature}: relatif {direction}"
        )

# 4. Confusion Matrix
print("\n4. INTERPRETASI CONFUSION MATRIX")

if labels is not None and not labels.isna().all():
    print(f"Accuracy setelah mapping = {accuracy:.4f}")

    if accuracy >= 0.90:
        print("Hasil: sangat baik.")
        print("Sebagian besar signal berhasil dikelompokkan")
        print("sesuai dengan label aslinya.")
    elif accuracy >= 0.75:
        print("Hasil: baik.")
        print("Mayoritas signal sudah berada pada kelompok yang sesuai.")
    elif accuracy >= 0.60:
        print("Hasil: cukup.")
        print("Masih terdapat beberapa signal yang tertukar.")
    else:
        print("Hasil: kurang baik.")
        print("Banyak signal masih tertukar antar kelas.")

    print("\nKecocokan setiap label:")

    for i, label in enumerate(true_labels):
        row_total = cm[i].sum()
        correct = cm[i, i]
        pct = correct / row_total * 100 if row_total else 0

        print(
            f"- {label}: {correct}/{row_total} "
            f"benar ({pct:.2f}%)"
        )
else:
    print("Confusion Matrix tidak dapat dibuat karena label asli")
    print("tidak tersedia.")

# 5. PCA
print("\n5. INTERPRETASI PCA")

explained = pca.explained_variance_ratio_

print(
    f"PC1 menjelaskan {explained[0] * 100:.2f}% variasi data."
)
print(
    f"PC2 menjelaskan {explained[1] * 100:.2f}% variasi data."
)
print(
    f"Total PC1 + PC2 = {explained.sum() * 100:.2f}% variasi data."
)

print("\nKESIMPULAN SIGNAL CLUSTERING")
print(
    f"Signal berhasil dikelompokkan menjadi {SELECTED_K} cluster "
    f"berdasarkan karakteristik statistik dan spektralnya."
)
print(
    f"Silhouette Score akhir adalah {final_silhouette:.4f}."
)